In [1]:
import os
import re


In [2]:



def load_resumes(resume_folder='resumes'):
    resumes = []
    filenames = []
    for filename in os.listdir(resume_folder):  # Loop through each file in the folder
        if filename.endswith('.txt'):  # Only process .txt files
            with open(os.path.join(resume_folder, filename), 'r', encoding='utf-8') as file:
                text = file.read()  # Read the entire text of the file
                resumes.append(text)  # Store the text of each resume
                filenames.append(filename)  # Store the name of each resume file
    return resumes, filenames

def clean_text(text):
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # Remove numbers and special characters
    text = re.sub(r'\s+', ' ', text)  # Replace multiple spaces with a single space
    return text.lower().strip()  # Convert to lowercase and remove leading/trailing spaces


In [3]:
raw_resumes, resume_names = load_resumes()  # Load the resumes from the folder
cleaned_resumes = [clean_text(resume) for resume in raw_resumes]  # Clean each resume


In [4]:
print(f"Loaded {len(cleaned_resumes)} resumes.")
print("Sample Resume (cleaned):\n", cleaned_resumes[0][:300])  # Print first 300 characters of the cleaned resume


Loaded 2 resumes.
Sample Resume (cleaned):
 john doe email johndoeexamplecom phone location new york ny objective to leverage my years of experience as a software engineer in a dynamic company that values innovation and teamwork skills programming languages python java c web development html css javascript react databases mysql mongodb versio


Loading and Cleaning the Job Description

In [5]:
def load_job_description(filepath='job.txt'):
    with open(filepath, 'r', encoding='utf-8') as file:
        text = file.read()
    return clean_text(text)  # Clean the job description


In [6]:
job_description = load_job_description()  # Load and clean the job description
print("Job Description (cleaned):\n", job_description[:300])  # Print first 300 characters of the cleaned job description


Job Description (cleaned):
 job title software engineer company tech innovations inc location remote job description we are looking for an experienced software engineer to join our team the ideal candidate should have a strong background in software development excellent problemsolving skills and the ability to collaborate in 


TF-IDF Vectorization

In [7]:

from sklearn.feature_extraction.text import TfidfVectorizer


In [8]:
# Combine job description and resumes for TF-IDF processing
all_docs = [job_description] + cleaned_resumes  # Combine job description and resumes

vectorizer = TfidfVectorizer()  # Create the TF-IDF model
tfidf_matrix = vectorizer.fit_transform(all_docs)  # Convert text into TF-IDF features


Cosine Similarity (Matching Resumes with Job Description)

In [9]:
from sklearn.metrics.pairwise import cosine_similarity


In [10]:
# Calculate similarity between job description and each resume
cosine_similarities = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()

# Sort by highest match
ranked_indices = cosine_similarities.argsort()[::-1]  # Sort the scores in descending order


In [11]:
# Print the results
print("\n🔍 Top Matching Resumes:")
for idx in ranked_indices:
    print(f"{resume_names[idx]} - Score: {cosine_similarities[idx]:.4f}")



🔍 Top Matching Resumes:
resume1.txt - Score: 0.4333
resume2.txt - Score: 0.1271


In [12]:
import pandas as pd

# Create a DataFrame to display results
results_df = pd.DataFrame({
    'Resume': [resume_names[i] for i in ranked_indices],
    'Similarity Score': [cosine_similarities[i] for i in ranked_indices]
})

results_df


,Resume,Similarity Score
0,resume1.txt,0.433349
1,resume2.txt,0.127056


In [13]:
!pip install transformers torch scikit-learn




[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip


In [14]:
!pip install sentence-transformers scikit-learn




[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: pip install --upgrade pip


In [17]:
from sentence_transformers import SentenceTransformer, util

# Load BERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Example job description
job_description = "Looking for a data scientist with experience in Python, machine learning, and data analysis."

# Example resumes (replace with your own list)
resumes = [
    "Experienced machine learning engineer with a focus on deep learning and Python.",
    "Frontend developer skilled in React, JavaScript, and UI design.",
    "Data analyst with strong skills in Python, Pandas, and data visualization.",
]

# Encode all texts
job_embedding = model.encode(job_description, convert_to_tensor=True)
resume_embeddings = model.encode(resumes, convert_to_tensor=True)

# Compute cosine similarity
cosine_scores = util.cos_sim(job_embedding, resume_embeddings)

# Print similarity scores
for i, score in enumerate(cosine_scores[0]):
    print(f"Resume {i+1} Similarity Score: {score.item():.4f}")
    



Resume 1 Similarity Score: 0.6942
Resume 2 Similarity Score: 0.2921
Resume 3 Similarity Score: 0.6815


In [18]:
from transformers import BertTokenizer, BertModel
import torch

# Load pre-trained BERT
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')


In [19]:
def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
    # Take the [CLS] token representation
    return outputs.last_hidden_state[:, 0, :].numpy()


In [20]:
# Example Job Description and Resumes
job_description = "Looking for a data scientist with experience in machine learning and Python."
resumes = [
    "Experienced in machine learning, data science, and Python development.",
    "Expert in HR and talent acquisition with no technical background.",
    "Proficient in Python, deep learning, and data analytics."
]

# Embeddings
job_desc_embedding = get_bert_embedding(job_description)  # Shape: (1, 768)
resume_embeddings = [get_bert_embedding(resume) for resume in resumes]  # List of arrays

# Convert list to single numpy array
import numpy as np
resume_embeddings = np.vstack(resume_embeddings)  # Shape: (3, 768)


In [21]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarities_bert = cosine_similarity(job_desc_embedding, resume_embeddings)

# Print scores
for i, score in enumerate(cosine_similarities_bert[0]):
    print(f"Resume {i+1} Similarity Score: {score:.4f}")


Resume 1 Similarity Score: 0.7949
Resume 2 Similarity Score: 0.7895
Resume 3 Similarity Score: 0.7845


#Model Evaluation and Testing

After obtaining the similarity scores, it's time to evaluate how well your model is performing.
What is the purpose here?

You need to test whether the BERT model's embeddings are correctly capturing the essence of the job descriptions and resumes.
We will use metrics like precision, recall, and F1-score to evaluate the performance.

In [22]:
from sklearn.metrics.pairwise import cosine_similarity

# Assume job_desc_embedding (1 x 768) and resume_embeddings (n x 768) already bana liye hain
cosine_similarities_bert = cosine_similarity(job_desc_embedding, resume_embeddings)

# Similarity scores ko print karo
for i, score in enumerate(cosine_similarities_bert[0]):
    print(f"Resume {i+1} Similarity Score: {score:.4f}")


Resume 1 Similarity Score: 0.7949
Resume 2 Similarity Score: 0.7895
Resume 3 Similarity Score: 0.7845


In [23]:
# Ground truth: 1 = relevant resume, 0 = not relevant
y_true = [1, 0, 1]


In [24]:
# Convert similarity scores to binary predictions
y_pred = [1 if score > 0.5 else 0 for score in cosine_similarities_bert[0]]

print("Predicted Labels:", y_pred)


Predicted Labels: [1, 1, 1]


In [25]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Precision, Recall, and F1
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)

print("Precision:", round(precision, 2))
print("Recall:", round(recall, 2))
print("F1 Score:", round(f1, 2))


Precision: 0.67
Recall: 1.0
F1 Score: 0.8


In [26]:
from transformers import BertTokenizer, BertModel
from sklearn.metrics.pairwise import cosine_similarity
import torch

# Load once at the start
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

def encode_text(text):
    tokens = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**tokens)
    return outputs.last_hidden_state.mean(dim=1)

def get_resume_scores(job_description, resumes, top_n=3):
    # Encode job description and all resumes
    job_desc_embedding = encode_text(job_description)
    resume_embeddings = torch.cat([encode_text(resume) for resume in resumes], dim=0)

    # Cosine similarity
    cosine_similarities = cosine_similarity(job_desc_embedding, resume_embeddings)

    # Get scores with indexes
    scored_resumes = list(enumerate(cosine_similarities[0]))
    scored_resumes.sort(key=lambda x: x[1], reverse=True)  # Sort by score descending

    return scored_resumes[:top_n]  # Return top N matches



In [27]:
job_desc = "Looking for a data scientist with NLP experience"
resumes = [
    "Experienced in machine learning and NLP projects",
    "Background in electrical engineering with no NLP",
    "Skilled in Python, BERT, and resume screening tools"
]

top_matches = get_resume_scores(job_desc, resumes)

for idx, score in top_matches:
    print(f"Resume {idx+1} Similarity Score: {score:.4f}")


Resume 1 Similarity Score: 0.7840
Resume 2 Similarity Score: 0.6811
Resume 3 Similarity Score: 0.6721
